# Curation of HES APC dataset


In [ ]:

import os
import sys

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 500)

In [ ]:
from util.general_utils import field_availability_check, list_field_instances
from util.dask_utils import import_field_from_dask
from util.parquet_maker import dask_to_parquet
from env.parameters import P
from util.logging import logging_setup

In [ ]:
import dask.dataframe as dd


In [ ]:
# load the cohort
df_cohort = pd.read_csv(f'''{P.output_csv_path}/cohort_1_base.csv''')
df_cohort.head()

# HES data

In [ ]:
path_hesin = f'''{P.output_parquet_path}/hesin'''
dd_hesin = dd.read_parquet(path_hesin)
dd_hesin.head()

In [ ]:
path_hesin_diag = f'''{P.output_parquet_path}/hesin_diag'''
dd_hesin_diag = dd.read_parquet(path_hesin_diag)
dd_hesin_diag.head()

In [ ]:
path_hesin_oper = f'''{P.output_parquet_path}/hesin_oper'''
dd_hesin_oper = dd.read_parquet(path_hesin_oper)
dd_hesin_oper.head()

In [ ]:
path_hesin_critical = f'''{P.output_parquet_path}/hesin_critical'''
dd_hesin_critical = dd.read_parquet(path_hesin_critical)
dd_hesin_critical.head()

In [ ]:
# load pandas and use them (hes files are not too large)
df_hesin = pd.read_csv(f'''{P.data_portal_path}/hesin.txt''', delimiter="\t")
df_hesin.head()

In [ ]:
df_diag = pd.read_csv(f'''{P.data_portal_path}/hesin_diag.txt''', delimiter="\t")
df_diag.head()

# Strategy

- hesin: includes episodes (one record per episode). Use for epistart, epiend, type, and duraiton of stay
    - No maternity
- hesin_diag: includes diagnoses per episode. Each row has one diagnosis in icd10
- Link hesin and hesin_diag on eid and ins_index
    in hes_diag, the diagnosis order is indicated by arr_index



# Test linking for phenotyping

In [ ]:
df_hesin_sel =df_hesin[['eid', 'ins_index', 'epistart', 'epiend']]

df_hesin_sel

In [ ]:
# some qc
df_diag[df_diag['diag_icd10'].isnull()]

In [ ]:
print(df_diag[df_diag['diag_icd10'].isnull()].shape)
print(df_diag[~df_diag['diag_icd10'].isnull()].shape)


In [ ]:
df_diag_sel =df_diag[['eid', 'ins_index', 'arr_index', 'level', 'diag_icd10']]
df_diag_sel

In [ ]:
df_hesin_diag = pd.merge(df_hesin_sel, df_diag_sel, on= ['eid', 'ins_index'], how='inner')
df_hesin_diag

In [ ]:
df_hesin_diag.shape

In [ ]:
# some qc
df_hesin_diag[df_hesin_diag['diag_icd10'].isnull()].shape

In [ ]:
df_hesin_diag[df_hesin_diag['epistart'].isnull()].shape


In [ ]:
df_hesin_diag[(df_hesin_diag['epistart'].isnull()) & (df_hesin_diag['epiend'].notnull())].shape
# Small number, no need for imputing

In [ ]:
df_hesin_diag_clean = df_hesin_diag[(df_hesin_diag['diag_icd10'].notnull()) & (df_hesin_diag['epistart'].notnull())]

df_hesin_diag_clean.shape

In [ ]:
# link with the cohort
df_j = pd.merge(df_cohort[['eid', 'dob', 'dod', 'evdt_ukb_end']], df_hesin_diag_clean, on= 'eid', how='left')
df_j.head()

In [ ]:
df_j.dtypes

In [ ]:

df_j['epistart'] = pd.to_datetime(df_j['epistart'], format= "%d/%m/%Y")
df_j['epiend'] = pd.to_datetime(df_j['epiend'], format= "%d/%m/%Y")
df_j

In [ ]:
# qc
df_j[df_j['dob']> df_j['epistart']].shape

In [ ]:
df_j[(df_j['dod'].notnull()) & (df_j['dod']<df_j['epistart'])].shape

In [ ]:
df_j[(df_j['dod'].notnull()) & (df_j['dod']<df_j['epistart'])].head()


In [ ]:
# negligable with only days of difference

In [ ]:
df_j[df_j['epistart']>= df_j['evdt_ukb_end']].shape


In [ ]:
df_j.shape

In [ ]:
df_j.to_csv(f'''{P.output_csv_path}/hesin_diag_curated.csv''', index=False)


In [ ]:
# save as parquet
dd_j = dd.from_pandas(df_j, npartitions=2)

dask_to_parquet(dd_j, output_path=P.output_parquet_path, output_parquet_name="hesin_diag_curated", write_index=False)
